FRED Flares

In [ ]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm

# --- Paths ---
sim_folder = r'file_path'
output_folder = r'file_path'

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

def inject_fred_flare_flux(time, mag_sim, amp_mag, t_peak, tau_rise, tau_decay):
    """
    Injects a FRED flare in flux space and converts back to magnitude.
    amp_mag: target peak brightening in magnitudes.
    """

    # 1. Convert baseline magnitude → flux
    flux_base = 10**(-0.4 * mag_sim)

    # 2. Compute peak flare flux from amp_mag
    median_flux = np.median(flux_base)
    peak_flux = median_flux * (10**(0.4 * amp_mag) - 1)

    # 3. FRED profile in flux space
    flare_flux = np.zeros_like(time)

    # ---------- Fast rise ----------
    rise_mask = time < t_peak
    x_rise = (time[rise_mask] - t_peak) / tau_rise

    rise_vals = np.zeros_like(x_rise)
    valid_rise = x_rise > -700   # below this exp ~ 0
    rise_vals[valid_rise] = np.exp(x_rise[valid_rise])

    flare_flux[rise_mask] = peak_flux * rise_vals

    # ---------- Exponential decay ----------
    decay_mask = time >= t_peak
    x_decay = -(time[decay_mask] - t_peak) / tau_decay

    decay_vals = np.zeros_like(x_decay)
    valid_decay = x_decay > -700
    decay_vals[valid_decay] = np.exp(x_decay[valid_decay])

    flare_flux[decay_mask] = peak_flux * decay_vals

    # 4. Convert back to magnitude
    total_flux = flux_base + flare_flux
    mag_with_flare = -2.5 * np.log10(total_flux)

    delta_mag = mag_sim - mag_with_flare

    return mag_with_flare, delta_mag


# ==========================================
# CONFIG
# ==========================================
files = [f for f in os.listdir(sim_folder) if f.endswith('.csv')]

for f in tqdm(files, desc="Injecting FRED flares"):

    df = pd.read_csv(os.path.join(sim_folder, f))
    time = df['MJD'].values

    # --- RANDOM PARAMETERS ---
    t_peak_mjd = np.random.choice(time)   # peak guaranteed visible

    amp = np.random.uniform(0.3, 1.2)
    tr  = np.random.uniform(10, 50)
    td  = np.random.uniform(100, 400)

    # Inject flare
    mag_flared, mag_signal = inject_fred_flare_flux(
        time,
        df['mag_r_sim'].values,
        amp,
        t_peak_mjd,
        tr,
        td
    )

    # Save
    df['mag_r_with_flare'] = mag_flared
    df['flare_mag_contribution'] = mag_signal

    df.to_csv(os.path.join(output_folder, f), index=False)

print(f"\nDone. {len(os.listdir(output_folder))} files saved.")


Gaussian Flares

In [ ]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm

# --- Paths ---
sim_folder = r'file_path'
output_folder = r'file_path'

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

def inject_gaussian_flare_flux(time, mag_sim, amp_mag, t_peak, sigma_days):
    """
    Injects a Gaussian flare in flux space and converts back to magnitude.
    amp_mag: target peak brightening in magnitudes.
    """

    # 1. Convert baseline magnitude → flux
    flux_base = 10**(-0.4 * mag_sim)

    # 2. Compute peak flare flux
    median_flux = np.median(flux_base)
    peak_flux = median_flux * (10**(0.4 * amp_mag) - 1)

    # 3. Gaussian profile in flux space
    x = (time - t_peak) / sigma_days

    gaussian_vals = np.zeros_like(x)
    valid = x**2 < 1400   # equivalent to exp(-700) cutoff
    gaussian_vals[valid] = np.exp(-0.5 * x[valid]**2)

    flare_flux = peak_flux * gaussian_vals

    # 4. Convert back to magnitude
    total_flux = flux_base + flare_flux
    mag_with_flare = -2.5 * np.log10(total_flux)

    delta_mag = mag_sim - mag_with_flare

    return mag_with_flare, delta_mag


# ==========================================
# CONFIG
# ==========================================
files = [f for f in os.listdir(sim_folder) if f.endswith('.csv')]

for f in tqdm(files, desc="Injecting Gaussian flares"):

    df = pd.read_csv(os.path.join(sim_folder, f))
    time = df['MJD'].values

    # --- RANDOM PARAMETERS ---
    # Peak guaranteed inside observed data
    t_peak_mjd = np.random.choice(time)

    amp = np.random.uniform(0.3, 1.2)      # Brightening amplitude
    sigma = np.random.uniform(20, 120)     # Gaussian width (days)

    # Inject flare
    mag_flared, mag_signal = inject_gaussian_flare_flux(
        time,
        df['mag_r_sim'].values,
        amp,
        t_peak_mjd,
        sigma
    )

    # Save
    df['mag_r_with_flare'] = mag_flared
    df['flare_mag_contribution'] = mag_signal

    df.to_csv(os.path.join(output_folder, f), index=False)

print(f"\nDone. {len(os.listdir(output_folder))} files saved.")


Gamma Flares

In [ ]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm

# --- Paths ---
sim_folder = r'file_path'
output_folder = r'file_path'

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

def inject_gamma_flare_flux(time, mag_sim, amp_mag, t_peak, k_shape, theta):
    """
    Injects a Gamma-profile flare in flux space.
    Peak is aligned to t_peak.
    """

    # 1. Baseline flux
    flux_base = 10**(-0.4 * mag_sim)

    # 2. Peak flare flux from magnitude amplitude
    median_flux = np.median(flux_base)
    peak_flux = median_flux * (10**(0.4 * amp_mag) - 1)

    # 3. Gamma profile
    # Shift so peak occurs at t_peak
    # peak time = (k-1)*theta
    t0 = t_peak - (k_shape - 1) * theta

    x = (time - t0)

    flare_flux = np.zeros_like(time)

    positive = x > 0
    x_pos = x[positive]

    # log-safe computation
    log_profile = (k_shape - 1) * np.log(x_pos) - x_pos / theta

    vals = np.zeros_like(x_pos)
    valid = log_profile > -700
    vals[valid] = np.exp(log_profile[valid])

    # normalize to unit peak
    if vals.max() > 0:
        vals /= vals.max()

    flare_flux[positive] = peak_flux * vals

    # 4. Convert back to magnitude
    total_flux = flux_base + flare_flux
    mag_with_flare = -2.5 * np.log10(total_flux)

    delta_mag = mag_sim - mag_with_flare

    return mag_with_flare, delta_mag


# ==========================================
# CONFIG
# ==========================================
files = [f for f in os.listdir(sim_folder) if f.endswith('.csv')]

for f in tqdm(files, desc="Injecting Gamma flares"):

    df = pd.read_csv(os.path.join(sim_folder, f))
    time = df['MJD'].values

    # Peak guaranteed inside observed data
    t_peak = np.random.choice(time)

    amp = np.random.uniform(0.3, 1.2)

    # Gamma shape parameters
    k_shape = np.random.uniform(2.0, 5.0)   # controls asymmetry
    theta   = np.random.uniform(20, 100)    # timescale

    mag_flared, mag_signal = inject_gamma_flare_flux(
        time,
        df['mag_r_sim'].values,
        amp,
        t_peak,
        k_shape,
        theta
    )

    df['mag_r_with_flare'] = mag_flared
    df['flare_mag_contribution'] = mag_signal

    df.to_csv(os.path.join(output_folder, f), index=False)

print(f"\nDone. {len(os.listdir(output_folder))} files saved.")


Single point spikes

In [ ]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm

# --- Paths ---
sim_folder = r'file_path'
output_folder = r'file_path'

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

def inject_single_spike_flux(time, mag_sim, amp_mag):
    """
    Injects a single-point spike in flux space.
    Only one existing data point is modified.
    """

    # 1. Convert baseline magnitude → flux
    flux_base = 10**(-0.4 * mag_sim)

    # 2. Choose random index
    idx = np.random.randint(0, len(time))

    # 3. Compute flare flux needed for desired mag brightening
    median_flux = np.median(flux_base)
    spike_flux = median_flux * (10**(0.4 * amp_mag) - 1)

    flare_flux = np.zeros_like(flux_base)
    flare_flux[idx] = spike_flux

    # 4. Convert back to magnitude
    total_flux = flux_base + flare_flux
    mag_with_spike = -2.5 * np.log10(total_flux)

    delta_mag = mag_sim - mag_with_spike

    return mag_with_spike, delta_mag


# ==========================================
# CONFIG
# ==========================================
files = [f for f in os.listdir(sim_folder) if f.endswith('.csv')]

for f in tqdm(files, desc="Injecting single-point spikes"):

    df = pd.read_csv(os.path.join(sim_folder, f))
    time = df['MJD'].values

    if len(time) < 2:
        continue

    # Random spike amplitude
    amp = np.random.uniform(0.3, 1.5)

    mag_spiked, mag_signal = inject_single_spike_flux(
        time,
        df['mag_r_sim'].values,
        amp
    )

    # Save
    df['mag_r_with_flare'] = mag_spiked
    df['flare_mag_contribution'] = mag_signal

    df.to_csv(os.path.join(output_folder, f), index=False)

print(f"\nDone. {len(os.listdir(output_folder))} files saved.")
